# LLC Object Editors

This notebook walks thru the financial workflow to update the general ledger (journal) and product periodic (monthly, YE) reports. 

Refer to [GHub> IRSGuides.ChartofAccounts.md](https://github.com/wbgroupmgr/LLC-WB-Group/blob/main/pages/IRSGuide/LLC-ChartOfAccoounts.md)

## Usage - refer to llcEditorCmd.py / ui.llcEditorCmd-UserGuide.md




In [1]:
# Load bookkeeping services : llc, coa, 
import os
from ledger.LLC import LLC
from pathlib import Path
import datetime
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from ledger.llcCOA import ChartOfAccounts
import pandas as pd



top = Path.cwd().parents[2]
llc = LLC('WBGroupLLC',debug=False, top=top)  # debug='details'
# Load Latest/YE Bank Stmt
llc._Bank()
# Chart of accounts
coa = ChartOfAccounts(llc)


<h2> Chart of Accounts

In [2]:
# Show COA for refernces
coa.toDF()
#coa.loadAll()['coaDict']

,acctID,acctDesc,acctType
Acct.Cash.Bank,1010,(Operating Bank Account: Cash ),Asset
Acct.Cash.Security,1020,Security Deposit Trust Account (Funds held for...,Asset
Acct.Fixed.Tangible.InConstruction,1310,Asset: Debit Buildings Not InService - purchas...,Asset
Acct.Fixed.Land,1410,Land : value of land,Asset
Acct.Fixed.Tangible,1420,Buildings (The purchase price of the physical ...,Asset
Acct.Fixed.Tangible.InService,1430,Building: Inservice value of building,Asset
Acct.Fixed.Depreciation.Accum,1460,"Accumulated Depreciation (A ""contra-asset"" acc...",Asset
Acct.Fixed.Tangible.Improvements,1470,"Buildings, Improvements (Major: new flooring, ...",Asset
Acct.Liab.Payable,2010,Accounts Payable (Unpaid bills to vendors),Liability
Acct.Liab.Morgage,2100,Mortgage Payable (The principal balance on you...,Liability


In [59]:
# Save ExpOnly json
import json
savFN = erObj.FN().replace('.json', '_expOnly.json')
with open(savFN, 'r') as fio:
    savList = json.load(fio)
savDF = pd.DataFrame(savList).drop(17)
tCols = ['tID', 'desc']
cols = ['acct','aType', 'Ledger']
savDF[tCols].head(1)

,tID,desc
0,2025.08.28_75.76,NoRcp: Approved Purchase: AMAZON MKTPL*6Y5UJ A...


In [ ]:
erList = erObj.load()
erDF = pd.DataFrame(erList)
expDF = erDF[erDF.acct.str.contains('Acct.Exp')]
xDF = erDF[~ erDF.acct.str.contains('Acct.Exp')]
print("Lens", len(xDF), len(expDF), len(savDF))
s1 = set(expDF.acct.unique())
s2 = set(savDF.acct.unique())
# Merge keys to find missing records
mDF = expDF[tCols].reset_index().merge(savDF[tCols],  on='tID', how='outer').sort_values(by='index')
mDF

In [81]:
mCols = ['refDoc', 'tID', 'desc']
m2DF = expDF[mCols].reset_index().merge(savDF[mCols],  on='refDoc', how='outer').sort_values(by='index')
m2DF

,index,refDoc,tID_x,desc_x,tID_y,desc_y
15,3,PURCHASE AUTHORIZED ON 08/27 AMAZON MKTPL*6Y5U...,2025.08.28_75.76,Approved Purchase,2025.08.28_75.76,NoRcp: Approved Purchase: AMAZON MKTPL*6Y5UJ A...
45,4,Pedernales Elect TEL PMTS 082825 1SNKYQ W B GR...,2025.08.28_250.0,Pay Monthly Util,2025.08.28_250.0,Auto: Pay Monthly Util
13,5,CHECK # 101,2025.09.03_150.0,Pay Monthly Util,2025.09.03_150.0,Auto: Pay Monthly Util
16,6,PURCHASE AUTHORIZED ON 09/10 Kings Feed And Ha...,2025.09.10_25.94,Approved Purchase,2025.09.10_25.94,Receipt+ Approved Purchase: Kings Feed And Har...
17,7,PURCHASE AUTHORIZED ON 09/19 AMAZON MKTPL*B108...,2025.09.22_38.94,Approved Purchase,2025.09.22_38.94,NoRecpt Approved Purchase: AMAZON MKTPL*B108S ...
18,9,PURCHASE AUTHORIZED ON 10/01 AMAZON MKTPL*NJ7X...,2025.10.03_8.29,Approved Purchase,2025.10.03_8.29,NoRcpt Approved Purchase: AMAZON MKTPL*NJ7XX A...
19,10,PURCHASE AUTHORIZED ON 10/03 WIMBERLEY ACE WIM...,2025.10.03_17.31,Approved Purchase,2025.10.03_17.31,NoRcpt Approved Purchase: WIMBERLEY ACE WIMBER...
20,11,PURCHASE AUTHORIZED ON 10/04 WAL-MART #0404 SA...,2025.10.06_140.73,Approved Purchase,NaN,NaN
14,12,CHECK # 102,2025.10.07_50.0,Pay Electrician Repair Outlet,2025.10.07_50.0,EPay Pay Electrician Repair Outlet
21,13,PURCHASE AUTHORIZED ON 10/06 LOWES #00907* 866...,2025.10.07_31.86,Approved Purchase,2025.10.07_31.86,NoRcpt Approved Purchase: LOWES #00907* 866-48...


In [94]:
nCols = ['dt', 'desc', 'amt', 'aType', 'acct', 'Ledger', 'acctSub', 'propNm', 'propID', 'propAddr', 'propOwners',
        'tID', 'tDB', 'refDB', 'refDoc']
tIDList = ['2025.10.06_140.73', '2025.10.09_14.06', '2025.10.08_27.04', '2025.10.08_51.06']
xxDF = expDF.iloc[[11,16,17,20]].copy()

newDF = pd.concat([xDF[nCols], savDF[nCols], xxDF[nCols]])
print(len(newDF), len(xDF), len(savDF), len(xxDF))
nList = newDF.sort_values(by='tID').reset_index(drop=True).to_dict(orient='records')
erObj.save(nList)


54 13 37 4


In [73]:
newDF = 


,dt,desc,amt,aType,acct,Ledger,acctSub,propNm,propID,propAddr,propOwners,tID,tDB,refDB,refDoc


In [58]:
savDF[savDF.tID == '2025.10.14_90.9']

,Ledger,aType,acct,acctSub,amt,desc,dt,propAddr,propID,propNm,propOwners,refDB,refDoc,tDB,tID,acctType,_unknown
11,Acct.Cash.Bank,Debit,Acct.Exp.Util,Waste,90.9,Auto: Approved Purchase: TEXAS DISPOSAL SYS 80...,2025.10.14,,e20250826-805HMD,H_805HighMesa,,llcBank,PURCHASE AUTHORIZED ON 10/13 TEXAS DISPOSAL SY...,llcBank,2025.10.14_90.9,Expense,
17,Acct.Cash.Bank,Debit,Acct.Exp.Other,Waste,90.9,Approved Purchase: TEXAS DISPOSAL SYS 800-3758375,2025.10.14,None,None,H_805HighMesa,None,llcBank,PURCHASE AUTHORIZED ON 10/13 TEXAS DISPOSAL SY...,llcBank,2025.10.14_90.9,Expense,NaN


## Load llcAssets/llcExpRev/llcFinancial, verify records pwer

In [3]:
# Test Initialize each LLC DB 

from ledger.llcAssets import llcAssets
aObj = llc.assets()

from ledger.llcExpRev import llcExpRev
erObj = llcExpRev(llc)
erObj.FN()

from ledger.llcFinancialReport import llcFinancialReport
frObj = llcFinancialReport(llc)
frObj.FN()

s =  f"- llcAssets: {len(aObj.load())}"
s += f"\n- llcExpRev: {len(erObj.load())}"
display(Markdown(s))

- llcAssets: 13
- llcExpRev: 38

In [4]:
from util.utilEditSession import utilEditSession
loadWk = True
if loadWk :
    es = utilEditSession(llcName='WBGroupLLC', load = True)
    es.loadTemp()
else:
    es = utilEditSession(llcName='WBGroupLLC', load = False)
aWk = es.get('llcAssets')
wk_aDF = aWk.toDF()
aWk.load()

wk_aDF.acctType.unique()
db_aDF = aWk.o.toDF()
db_aDF['acctType'] = db_aDF.acct.apply(lambda v : coa._Type(v))
db_aDF.acctType.unique()

#diff = DeepDiff(dict_a, dict_b)

99 eSession bind objects: {'wkAssets': '/tmp/llcAssets_WBGroupLLC_temp.json', 'wkExpRev': '/tmp/llcExpRev_WBGroupLLC_temp.json'}


array(['Asset', 'Equity', 'Expense'], dtype=object)

## Editor Status:  WK vs DB

In [5]:
def toAcctType(tObj):
    tList = tObj.load()
    return pd.DataFrame(tObj.llc.coa.getAcctType(tList))
    
es.clean()
es.loadTemp()

_aWk = es.get('llcAssets')
_erWk = es.get('llcExpRev')

_aDF = _aWk.toDF()
_erDF = _erWk.toDF
_erWk.loadTemp()

print("---- transaction acctType WK :: DB ----")

print(f"\nllcAsset WK acctTypes: {_aWk.toDF().acctType.unique()}")
print(f"llcAsset DB acctTypes: {toAcctType(_aWk.o).acctType.unique()}")
print(f"\nllcExpRev WK acctTypes: {_erWk.toDF().acctType.unique()}")
print(f"llcExpRev DB acctTypes: {toAcctType(_erWk.o).acctType.unique()}")

print("---- transaction acct WK :: DB ----")

print(f"\nllcAsset WK acct: {_aWk.toDF().acct.unique()}")
print(f"llcAsset DB acct: {_aWk.o.toDF().acct.unique()}")
print(f"\nllcExpRev WK acct: {_erWk.toDF().acct.unique()}")
print(f"llcExpRev DB acct: {_erWk.o.toDF().acct.unique()}")

print("\n---- transaction Assets transaction diff ----")
dif = (_aWk == _aWk.o)
if dif is True:
    print("aWk::DB Records Equal")
elif dif is True:
    print("aWk::DB Diff : number of records")
elif isinstance(dif, dict):
    print("aWk::DB Record differences")
    for k,v in dif.items():
        print(k, v)
_aDif = dif

print("\n---- transaction ExpRev transaction diff ----")
dif = (_erWk == _erWk.o)
if dif is True:
    print("aWk::DB Records Equal")
elif dif is True:
    print("aWk::DB Diff : number of records")
elif isinstance(dif, dict):
    print("erWk::DB Record differences")
    for k,v in dif.items():
        print(k, v)
_erDif = dif

---- transaction acctType WK :: DB ----

llcAsset WK acctTypes: ['Equity' 'Asset']
llcAsset DB acctTypes: ['Equity' 'Asset']

llcExpRev WK acctTypes: ['Asset']
llcExpRev DB acctTypes: ['Asset']
---- transaction acct WK :: DB ----

llcAsset WK acct: ['Acct.Cash.Bank' 'Acct.Equity.Owner.Capital.Funds'
 'Acct.Exp.Depreciation']
llcAsset DB acct: ['Acct.Cash.Bank' 'Acct.Equity.Owner.Capital.Funds'
 'Acct.Exp.Depreciation']

llcExpRev WK acct: ['Acct.Exp.Other' 'Acct.Exp.Util' 'Acct.Exp.Repair']
llcExpRev DB acct: ['Acct.Exp.Other' 'Acct.Exp.Util' 'Acct.Exp.Repair']

---- transaction Assets transaction diff ----
aWk::DB Record differences
0 {'values_changed': {"root['acctType']": {'new_value': 'Asset', 'old_value': 'Equity'}}}
1 {'values_changed': {"root['acctType']": {'new_value': 'Asset', 'old_value': 'Equity'}}}
2 {'values_changed': {"root['acctType']": {'new_value': 'Asset', 'old_value': 'Equity'}}}
5 {'values_changed': {"root['acctType']": {'new_value': 'Equity', 'old_value': 'Asset'}}

In [6]:
# def fixAssetFields
def fixAssetFields(tObj):
    df = tObj.toDF()
    oCols = ['dt', 'desc', 'amt', 'aType', 'acct', 'Ledger', 'acctSub', 'propNm', 'propID', 'propAddr', 'propOwners',
           'tID', 'tDB', 'refDB', 'refDoc']
    tObj.o.save(df[oCols].to_dict(orient='records'))
#fixAssetFields(_aWk)
#_aWk.loadTemp()
#display(_aWk.toDF().head(4))
#display(_aWk.o.toDF().head(4))

In [4]:
# Tst functions
def TestWk(testNm, loadOpt=False):
    print()
    display(Markdown(f"{'_'*10} {testNm}"))

    display(Markdown("<h3> Init EditSession"))
    es = utilEditSession(llcName='WBGroupLLC', load = loadOpt)
    
    display(Markdown("<h4> Assert 0 recoreds in working, otherwise clean"))
    # Assert 0 working files
    display(es)
    
    display(Markdown("<h2>Test loadTemp w/ loadOpt = False"))
    es.loadTemp()
    
    display(Markdown("<h4> Assert 0 recoreds in working, no loads"))
    # load failse because loadOpt set to False
    display(es)

    return es


## Test utilWorkingDB for each DB

In [5]:
from util.utilEditSession import utilEditSession

_ = TestWk("\n<h2>1. Test init", loadOpt=False)
es = TestWk("\n<h2>2. Test Init w/ Load=True", loadOpt=True)
es.clean()

__________ 
<h2>1. Test init

<h3> Init EditSession

99 eSession bind objects: {'wkAssets': '/tmp/llcAssets_WBGroupLLC_temp.json', 'wkExpRev': '/tmp/llcExpRev_WBGroupLLC_temp.json'}


<h4> Assert 0 recoreds in working, otherwise clean

llcEditSession Status:
Work oID:wkAssets> num:13:13, st:NoChanges, Load:False,  edOpt:llc wFN:/tmp/llcAssets_WBGroupLLC_temp.json,  
Work oID:wkExpRev> num:54:54, st:NoChanges, Load:False,  edOpt:llc wFN:/tmp/llcExpRev_WBGroupLLC_temp.json,  

...

<h4> Assert 0 recoreds in working, otherwise clean

llcEditSession Status:
Work oID:wkAssets> num:13:13, st:NoChanges, Load:True,  edOpt:llc wFN:/tmp/llcAssets_WBGroupLLC_temp.json,  
Work oID:wkExpRev> num:54:54, st:NoChanges, Load:True,  edOpt:llc wFN:/tmp/llcExpRev_WBGroupLLC_temp.json,  

<h2>Test loadTemp w/ loadOpt = False

<h4> Assert 0 recoreds in working, no loads

llcEditSession Status:
Work oID:wkAssets> num:13:13, st:NoChanges, Load:True,  edOpt:llc wFN:/tmp/llcAssets_WBGroupLLC_temp.json,  
Work oID:wkExpRev> num:54:54, st:NoChanges, Load:True,  edOpt:llc wFN:/tmp/llcExpRev_WBGroupLLC_temp.json,  

In [6]:
es

llcEditSession Status:
Work oID:wkAssets> num:0:13, st:NoChanges, Load:True,  edOpt:llc wFN:/tmp/llcAssets_WBGroupLLC_temp.json,  
Work oID:wkExpRev> num:0:54, st:NoChanges, Load:True,  edOpt:llc wFN:/tmp/llcExpRev_WBGroupLLC_temp.json,  

In [10]:
wk = es.oDict['llcExpRev']
wk.FN(), wk.o.FN()

('/tmp/llcExpRev_WBGroupLLC_temp.json',
 '/Users/frankrojas/GDrive/Family/Assets-Hobby/RealEstateInvestments/LLC-WB-Group/pages/AccountingData/Accts/llcExpRev_WBGroupLLC.json')

## Manual Fix for llcExpRev DB

In [47]:
#es.loadTemp()
import re

def _delFld(d, fldList):
    for f in fldList:
        if f in d : del d['acctType']
    return d

def toDesc(r):
    
    s = r.refDoc
    if pd.isna(s) : return s
        
    pat = r'ON \S* (.*)'
    m = re.search(pat, s)
    if m : 
        return(r.desc +': ' + ' '.join(m.group(1).split()[0:4]))
    else:
        return r.desc

def fixExp(tDict):
    # change Acct.Exp : acct='Acct.Cash.Bank', ledger=r.acct, aType = 'Credit'
    isExp = 'Acct.Exp' in tDict['acct']
    if  isExp and tDict['aType'] == 'Credit':
        print(f"acct   is Exp, aType;{tDict['aType']}, acct:{tDict['acct']}, Ledger:{tDict['Ledger']} -- changed")
        a = tDict['acct']
        tDict['acct'] = tDict['Ledger']
        tDict['Ledger'] = a
        
    elif isExp:
        print(f"acct   is Exp, aType;{tDict['aType']}, acct:{tDict['acct']}, Ledger:{tDict['Ledger']} -- swapped")
        #print("----", tDict['desc'])
        a = tDict['acct']
        tDict['acct'] = tDict['Ledger']
        tDict['Ledger'] = a
    elif 'Acct.Exp' in tDict['Ledger']:
        print(f"Ledger is Exp, aType;{tDict['aType']}, acct:{tDict['acct']}, Ledger:{tDict['Ledger']}")
        print("----", tDict['desc'])
    return tDict
        
        
def fixDesc(erObj):

    # Fix Acct.Exp posts: credit Cash.bank; debit Acct.Exp ; except aType='Debit' for returns
    newList = [fixExp(tDict) for tDict in erObj.load()]
    newDF = pd.DataFrame(newList)[coa.recCols()]


    erList = erObj.load()
    print("wList:", len(erList))
    erDF = pd.DataFrame(erList)
    erCols = list(erDF.columns)
    
    
    erDF.desc 
    erDF['wDesc'] = erDF.apply(lambda r : toDesc(r), axis=1)
    # Check if putput is good
    #erDF[['dt','amt','acct','Ledger', 'desc', 'wDesc', 'refDoc']]
    erDF.desc = erDF.wDesc
    erDF.drop(columns = ['wDesc'], inplace=True)
    
erObj = es.oDict['llcExpRev']
erList = erObj.load()
print("wList:", len(erList))
erDF = pd.DataFrame(erList)

#erObj.save(.to_dict(orient='records'))
erDF

wList: 38


,Ledger,aType,acct,acctSub,amt,desc,dt,propAddr,propID,propNm,propOwners,refDB,refDoc,tDB,tID,acctType,_unknown
0,Acct.Cash.Bank,Credit,Acct.Exp.Other,None,75.76,Approved Purchase: AMAZON MKTPL*6Y5UJ Amzn.com...,2025.08.28,None,None,H_805HighMesa,None,llcBank,PURCHASE AUTHORIZED ON 08/27 AMAZON MKTPL*6Y5U...,llcBank,2025.08.28_75.76,Expense,NaN
1,Acct.Cash.Bank,Credit,Acct.Exp.Util,Elec,250.00,Pay Monthly Util,2025.08.28,None,None,H_805HighMesa,None,llcBank,Pedernales Elect TEL PMTS 082825 1SNKYQ W B GR...,llcBank,2025.08.28_250.0,Expense,NaN
2,Acct.Cash.Bank,Credit,Acct.Exp.Util,Water,150.00,Pay Monthly Util,2025.09.03,None,None,H_805HighMesa,None,llcBank,CHECK # 101,llcBank,2025.09.03_150.0,Expense,NaN
3,Acct.Cash.Bank,Credit,Acct.Exp.Other,,25.94,"Approved Purchase: Kings Feed And Hardware, pl...",2025.09.10,,,H_805HighMesa,,llcBank,PURCHASE AUTHORIZED ON 09/10 Kings Feed And Ha...,llcBank,2025.09.10_25.94,Expense,
4,Acct.Cash.Bank,Credit,Acct.Exp.Other,None,38.94,Approved Purchase: AMAZON MKTPL*B108S Amzn.com...,2025.09.22,None,None,H_805HighMesa,None,llcBank,PURCHASE AUTHORIZED ON 09/19 AMAZON MKTPL*B108...,llcBank,2025.09.22_38.94,Expense,NaN
5,Acct.Cash.Bank,Credit,Acct.Exp.Other,None,8.29,Approved Purchase: AMAZON MKTPL*NJ7XX Amzn.com...,2025.10.03,None,None,H_805HighMesa,None,llcBank,PURCHASE AUTHORIZED ON 10/01 AMAZON MKTPL*NJ7X...,llcBank,2025.10.03_8.29,Expense,NaN
6,Acct.Cash.Bank,Credit,Acct.Exp.Other,None,17.31,Approved Purchase: WIMBERLEY ACE WIMBERLEY TX,2025.10.03,None,None,H_805HighMesa,None,llcBank,PURCHASE AUTHORIZED ON 10/03 WIMBERLEY ACE WIM...,llcBank,2025.10.03_17.31,Expense,NaN
7,Acct.Cash.Bank,Credit,Acct.Exp.Util,Util,50.00,Pay Electrician Repair Outlet,2025.10.07,None,None,H_805HighMesa,None,llcBank,CHECK # 102,llcBank,2025.10.07_50.0,Expense,NaN
8,Acct.Cash.Bank,Credit,Acct.Exp.Other,None,31.86,Approved Purchase: LOWES #00907* 866-483-7521 NC,2025.10.07,None,None,H_805HighMesa,None,llcBank,PURCHASE AUTHORIZED ON 10/06 LOWES #00907* 866...,llcBank,2025.10.07_31.86,Expense,NaN
9,Acct.Cash.Bank,Credit,Acct.Exp.Repair,,14.06,Approved Purchase - ACE Garbage Disposal key -...,2025.10.07,,e20250826-805HMD,H_805HighMesa,,llcBank,PURCHASE AUTHORIZED ON 10/07 WIMBERLEY ACE WIM...,llcBank,2025.10.07_14.06,Expense,


In [35]:
coa.recCols()

['dt',
 'desc',
 'amt',
 'aType',
 'acct',
 'Ledger',
 'acctSub',
 'propNm',
 'propID',
 'propAddr',
 'propOwners',
 'tID',
 'tDB',
 'refDB',
 'refDoc']

In [20]:
for wkID, wk  in es.oDict.items():
    display(Markdown(f"<h2> Test Wk.Load - {wkID}"))
    
    wkDF = pd.DataFrame(wk.load())
    display(Markdown(f"<h2> Test {wk.oID} Load"))
    display(wkDF.head(3))
    print("\n-----------\n")
    oDF = pd.DataFrame(wk.o.load())
    display(Markdown(f"<h2> Test {wk.o.oID} Load"))
    display(oDF.head(3))
    
            
    

<h2> Test Wk.Load - llcAssets

<h2> Test wkAssets Load

""



-----------



<h2> Test llcAssets Load

...

<h2> Test wkExpRev Load

,dt,desc,amt,aType,acct,Ledger,acctMajor,acctMinor,acctSub,propNm,propID,propAddr,propOwners,tID,tDB,refDB,refDoc,_unknown
0,2025.08.20,Owner investment,219000.00,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,None,None,None,H_805HighMesa,None,None,None,2025.08.20_219000.0,llcBank,llcBank,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...,{}
1,2025.08.20,Owner Investment,50.00,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,None,None,None,H_805HighMesa,None,None,None,2025.08.20_50.0,llcBank,llcBank,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...,{}
2,2025.08.26,Property Purchase,213936.95,Credit,Acct.Cash.Bank,Acct.Fixed.Tangible.InService,None,None,None,H_805HighMesa,None,None,None,2025.08.26_213936.95,llcBank,llcBank,WITHDRAWAL MADE IN A BRANCH/STORE,{}



-----------



<h2> Test llcExpRev Load

,dt,desc,amt,aType,acct,Ledger,acctMajor,acctMinor,acctSub,propNm,propID,propAddr,propOwners,tID,tDB,refDB,refDoc,_unknown
0,2025.08.20,Owner investment,219000.00,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,NaN,NaN,NaN,H_805HighMesa,NaN,NaN,NaN,2025.08.20_219000.0,llcBank,llcBank,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...,{}
1,2025.08.20,Owner Investment,50.00,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,NaN,NaN,NaN,H_805HighMesa,NaN,NaN,NaN,2025.08.20_50.0,llcBank,llcBank,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...,{}
2,2025.08.26,Property Purchase,213936.95,Credit,Acct.Cash.Bank,Acct.Fixed.Tangible.InService,NaN,NaN,NaN,H_805HighMesa,NaN,NaN,NaN,2025.08.26_213936.95,llcBank,llcBank,WITHDRAWAL MADE IN A BRANCH/STORE,{}


In [ ]:
stop

In [ ]:
es.clean()

## Load Temp files for Editors

In [ ]:
  # Load Notebook Editor
from util.uiEditors import TransactionEditor
from ledger.llcExpRev import llcExpRev

from util.uiEditors import nbAssetEditor, nbExpenseEditor

aObj = llc.assets()
erObj = llcExpRev(llc)

aEd = nbAssetEditor(aObj)
erEd = nbExpenseEditor(erObj)

aList = aObj.load()
aEd.saveTemp(aList)

erList = erObj.load()
erEd.saveTemp(erList)
print(f"Asset: {len(aList)} at FN:{aEd.oFN}")
print(f"ExpRev {len(erList)} at FN:{erEd.oFN}")

## Start Editor - both | assets | exprev

In [ ]:
edOpt = 'assets'
if edOpt == 'both':
    ed = TransactionEditor(
        asset_file=aEd.oFN,
        exprev_file=erEd.oFN
    )
    # Start the editor with debug mode
    
elif edOpt == 'assets':
    ed = TransactionEditor(asset_file=aEd.oFN)
elif edOpt == 'exprev':
    ed = TransactionEditor(exprev_file=erEd.oFN)

ed.start(debug=True) #port=5000, height=800, debug=True)
   




In [ ]:
aObj.load()


In [ ]:
ed = nbAssetEditor(llc.assets())
editMaster = True
if editMaster:
    # Edit Master llcAsset DB ------ move master into temp
    mList = ed.o.load() #len(aList)
    ed.saveTemp(mList)
    ed.editTemp()
else:
    # Temp has not been saved
    ed.editTemp()

## Start Asset Editor

In [ ]:
# Load Notebook Editor
from util.uiEditors import nbAssetEditor

ed = nbAssetEditor(llc.assets())
editMaster = True
if editMaster:
    # Edit Master llcAsset DB ------ move master into temp
    mList = ed.o.load() #len(aList)
    ed.saveTemp(mList)
    ed.editTemp()
else:
    # Temp has not been saved
    ed.editTemp()

In [ ]:
stop

In [ ]:
ed.stop()

In [ ]:
if True:
    # Manually update Master w/ temp
    # Do this once reconcilation is done
    mList = ed.loadTemp()
    ed.o.save(mList)

In [ ]:
ed.o.FN()

In [ ]:
ed.stop()